In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
dataset=pd.read_csv('Social_Network_Ads.csv')

In [3]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [4]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [6]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [10]:
dataset.drop('User ID',axis=1)

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [12]:
indep=dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dep=dataset[['Purchased']]

In [14]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(indep,dep,random_state=0,test_size=1/3)

In [21]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
param_grid ={ 'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'max_features': ['sqrt', 'log2', None]}
grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train, y_train) 
 

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(estimator=DecisionTreeClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 10, None],
                         'max_features': ['sqrt', 'log2', None]},
             scoring='f1_weighted', verbose=3)

In [23]:
print(grid.best_params_)

{'criterion': 'gini', 'max_depth': 3, 'max_features': None}


In [24]:
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(X_test) 

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)


In [25]:
cm

array([[78,  7],
       [ 7, 42]], dtype=int64)

In [26]:
print(clf_report)

              precision    recall  f1-score   support

           0       0.92      0.92      0.92        85
           1       0.86      0.86      0.86        49

    accuracy                           0.90       134
   macro avg       0.89      0.89      0.89       134
weighted avg       0.90      0.90      0.90       134



In [27]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])

0.8948379351740696

In [28]:
table=pd.DataFrame.from_dict(re)

In [29]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_max_features,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004956,0.001433,0.028453,0.007816,gini,3,sqrt,"{'criterion': 'gini', 'max_depth': 3, 'max_fea...",0.847141,0.793754,0.753180,0.885265,0.981233,0.852115,0.078727,17
1,0.013302,0.005085,0.032962,0.009345,gini,3,log2,"{'criterion': 'gini', 'max_depth': 3, 'max_fea...",0.799620,0.793754,0.870362,0.841025,0.981014,0.857155,0.067979,16
2,0.021458,0.003966,0.028465,0.005318,gini,3,None,"{'criterion': 'gini', 'max_depth': 3, 'max_fea...",0.869532,0.870047,0.870362,0.944161,0.981233,0.907067,0.046911,1
3,0.011916,0.003869,0.025572,0.004291,gini,5,sqrt,"{'criterion': 'gini', 'max_depth': 5, 'max_fea...",0.847141,0.723353,0.773585,0.943699,0.923510,0.842258,0.084582,19
4,0.006702,0.001501,0.019695,0.002710,gini,5,log2,"{'criterion': 'gini', 'max_depth': 5, 'max_fea...",0.847141,0.765553,0.869709,0.851527,0.859176,0.838621,0.037328,22
5,0.006149,0.002102,0.018118,0.003549,gini,5,None,"{'criterion': 'gini', 'max_depth': 5, 'max_fea...",0.787654,0.868752,0.832483,0.906166,0.923510,0.863713,0.049323,11
6,0.003468,0.002193,0.016807,0.007814,gini,10,sqrt,"{'criterion': 'gini', 'max_depth': 10, 'max_fe...",0.804764,0.889022,0.796284,0.832483,0.885265,0.841563,0.039113,20
7,0.004063,0.003647,0.017398,0.003142,gini,10,log2,"{'criterion': 'gini', 'max_depth': 10, 'max_fe...",0.804764,0.870047,0.796284,0.869709,0.865054,0.841172,0.033344,21
8,0.006579,0.001170,0.015651,0.001347,gini,10,None,"{'criterion': 'gini', 'max_depth': 10, 'max_fe...",0.849794,0.849057,0.833323,0.870362,0.943041,0.869115,0.038790,8
9,0.005470,0.002194,0.016315,0.003105,gini,None,sqrt,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.804764,0.885035,0.814409,0.814409,0.811321,0.825988,0.029733,24


In [30]:
age_input=float(input("Age:"))
salary_input=float(input("BMI:"))
sex_male_input=int(input("Sex Male 0 or 1:"))

Age: 40
BMI: 40000
Sex Male 0 or 1: 0


In [31]:
Social_network_Ads_Prediction=grid.predict([[age_input,salary_input,sex_male_input]])
print("Social_network_Ads_Prediction={}".format(Social_network_Ads_Prediction))

Social_network_Ads_Prediction=[1]
